# Aula 05 — Documents, Metadados e Busca Vetorial com LangChain

## Exercício 1 — Criando Documents manualmente

Nesta etapa são criados objetos `Document` manualmente para compreender a estrutura padrão utilizada pelo LangChain.

Cada `Document` possui dois componentes principais:

- `page_content`: conteúdo textual;
- `metadata`: informações adicionais associadas ao documento.

In [1]:
from langchain_core.documents import Document

In [2]:
documentos = [
    Document(
        page_content="Embeddings são representações vetoriais densas de textos.",
        metadata={
            "fonte": "embeddings.md",
            "pagina": 1,
            "tipo": "teoria",
            "tema": "embeddings",
            "autor": "Maria Eduarda Ribeiro da Silva"
        }
    ),

    Document(
        page_content="A similaridade de cosseno pode ser utilizada para comparar embeddings.",
        metadata={
            "fonte": "embeddings.md",
            "pagina": 2,
            "tipo": "pratica",
            "tema": "embeddings",
            "autor": "Maria Eduarda Ribeiro da Silva"
        }
    ),

    Document(
        page_content="Chunking consiste em dividir documentos grandes em trechos menores.",
        metadata={
            "fonte": "chunking.md",
            "pagina": 1,
            "tipo": "teoria",
            "tema": "chunking",
            "autor": "Maria Eduarda Ribeiro da Silva"
        }
    ),

    Document(
        page_content="RAG combina recuperação de informações com geração de texto por modelos de linguagem.",
        metadata={
            "fonte": "rag.md",
            "pagina": 1,
            "tipo": "teoria",
            "tema": "rag",
            "autor": "Maria Eduarda Ribeiro da Silva"
        }
    ),

    Document(
        page_content="Tokenização divide textos em unidades menores chamadas tokens.",
        metadata={
            "fonte": "tokenizacao.md",
            "pagina": 1,
            "tipo": "teoria",
            "tema": "tokenizacao",
            "autor": "Maria Eduarda Ribeiro da Silva"
        }
    )
]

In [3]:
print("Quantidade de documentos:", len(documentos))

Quantidade de documentos: 5


In [4]:
for i, documento in enumerate(documentos, start=1):
    print("=" * 70)
    print(f"DOCUMENTO {i}")
    print("Conteúdo:")
    print(documento.page_content)

    print("\nMetadados:")
    print(documento.metadata)

DOCUMENTO 1
Conteúdo:
Embeddings são representações vetoriais densas de textos.

Metadados:
{'fonte': 'embeddings.md', 'pagina': 1, 'tipo': 'teoria', 'tema': 'embeddings', 'autor': 'Maria Eduarda Ribeiro da Silva'}
DOCUMENTO 2
Conteúdo:
A similaridade de cosseno pode ser utilizada para comparar embeddings.

Metadados:
{'fonte': 'embeddings.md', 'pagina': 2, 'tipo': 'pratica', 'tema': 'embeddings', 'autor': 'Maria Eduarda Ribeiro da Silva'}
DOCUMENTO 3
Conteúdo:
Chunking consiste em dividir documentos grandes em trechos menores.

Metadados:
{'fonte': 'chunking.md', 'pagina': 1, 'tipo': 'teoria', 'tema': 'chunking', 'autor': 'Maria Eduarda Ribeiro da Silva'}
DOCUMENTO 4
Conteúdo:
RAG combina recuperação de informações com geração de texto por modelos de linguagem.

Metadados:
{'fonte': 'rag.md', 'pagina': 1, 'tipo': 'teoria', 'tema': 'rag', 'autor': 'Maria Eduarda Ribeiro da Silva'}
DOCUMENTO 5
Conteúdo:
Tokenização divide textos em unidades menores chamadas tokens.

Metadados:
{'fonte':

In [5]:
documento_teste = Document(
    page_content="Documento utilizado para testar diferentes tipos de metadados.",
    metadata={
        "fonte": "teste.md",
        "tags": ["LangChain", "Document", "metadata"],
        "informacoes": {
            "curso": "Residência em IA",
            "aula": 5
        },
        "ativo": True,
        "nota": 10
    }
)

print(documento_teste.metadata)

{'fonte': 'teste.md', 'tags': ['LangChain', 'Document', 'metadata'], 'informacoes': {'curso': 'Residência em IA', 'aula': 5}, 'ativo': True, 'nota': 10}


### Resposta — tipos aceitos em metadata

O campo `metadata` aceita diferentes tipos de dados Python, como strings, números, valores booleanos, listas e dicionários aninhados.

No teste realizado, foi possível armazenar tanto uma lista de tags quanto um dicionário contendo informações adicionais.

Entretanto, embora o objeto `Document` permita essas estruturas, algumas vector stores podem impor limitações aos tipos aceitos para filtragem de metadados. Por isso, em aplicações reais, costuma ser conveniente utilizar valores simples quando os metadados serão utilizados como filtros.

In [6]:
documento_sem_metadata = Document(
    page_content="Este documento foi criado sem informar metadata."
)

print("Conteúdo:", documento_sem_metadata.page_content)
print("Metadata:", documento_sem_metadata.metadata)

Conteúdo: Este documento foi criado sem informar metadata.
Metadata: {}


### Resposta — Document sem metadata

Quando um `Document` é criado sem fornecer o campo `metadata`, o LangChain cria automaticamente um dicionário vazio `{}`.

Portanto, `metadata` não é obrigatório para a criação do objeto.

### Campos adicionais escolhidos

**`embedding_model`**

Permite identificar qual modelo de embedding foi utilizado para representar aquele chunk. Esse campo é útil caso diferentes modelos sejam comparados futuramente.

**`secao`**

Permite rastrear a seção ou heading do documento de onde o conteúdo foi extraído. Isso pode ajudar tanto na filtragem quanto na apresentação da fonte durante uma resposta de RAG.

**`subdividido`**

Indica se o chunk original precisou passar por uma subdivisão adicional. Esse campo é útil para identificar trechos que excederam limites de tamanho durante o processamento.

**`aula_origem`**

Permite registrar em qual etapa do projeto o dado foi gerado. Isso facilita a rastreabilidade e a organização do pipeline experimental.

## Exercício 2 — Projetando o schema de metadados

| Campo | Tipo | Descrição |
|---|---|---|
| fonte | string | Nome do arquivo `.md` de origem |
| documento_id | string | Identificador do documento |
| chunk_index | int | Posição do chunk dentro do documento |
| estrategia | string | Estratégia de chunking utilizada |
| chunk_size | int ou null | Tamanho configurado para o chunk |
| chunk_overlap | int | Sobreposição configurada |
| n_caracteres | int | Quantidade real de caracteres do chunk |
| secao | string ou null | Seção do documento de onde o chunk foi extraído |
| subdividido | bool | Indica se o chunk precisou ser subdividido |
| idioma | string | Idioma predominante do chunk |

### Justificativa dos campos adicionais

**secao**

Permite identificar em qual parte do documento o chunk estava localizado, como Introdução, Metodologia ou Conclusão. Esse campo pode ajudar a localizar a informação original e também pode ser utilizado em filtros.

**subdividido**

Permite saber se o chunk original precisou ser dividido novamente por ultrapassar o limite de tamanho aceito pelo modelo de embeddings. Isso ajuda a rastrear modificações feitas durante o processamento.

**idioma**

Permite identificar o idioma predominante do chunk. Esse campo pode ser útil para filtrar resultados de busca ou trabalhar com documentos multilíngues.

In [8]:
exemplo_chunk = {
    "fonte": "bioetica_e_ia.md",
    "documento_id": "doc03",
    "chunk_index": 12,
    "estrategia": "recursive",
    "chunk_size": 1000,
    "chunk_overlap": 100,
    "n_caracteres": 823,
    "secao": "Autonomia e opacidade algorítmica",
    "subdividido": False,
    "idioma": "pt"
}

exemplo_chunk

{'fonte': 'bioetica_e_ia.md',
 'documento_id': 'doc03',
 'chunk_index': 12,
 'estrategia': 'recursive',
 'chunk_size': 1000,
 'chunk_overlap': 100,
 'n_caracteres': 823,
 'secao': 'Autonomia e opacidade algorítmica',
 'subdividido': False,
 'idioma': 'pt'}

### Qual campo incluir para citar a fonte em uma resposta de RAG?

Para permitir uma citação mais precisa da origem da informação, eu utilizaria os campos `fonte`, `secao` e, se disponível, `pagina`.

O campo `fonte` identifica o documento original, enquanto `secao` e `pagina` ajudam a localizar exatamente onde a informação foi encontrada.

### Por que chunk_index é útil?

O campo `chunk_index` indica a posição do chunk dentro do documento original.

Ele é útil quando um trecho recuperado está cortado no meio de uma explicação. Nesse caso, é possível buscar o chunk anterior ou o próximo usando o índice, recuperando contexto adicional sem precisar processar novamente o documento inteiro.